# Import Libraries

In [25]:
import requests
import pandas as pd
from datetime import datetime, timedelta

# API function

In [26]:
def get_carbon_intensity(start_date, end_date):
    url = f"https://api.carbonintensity.org.uk/intensity/{start_date}/{end_date}"
    response = requests.get(url)
    data = response.json()
    df = pd.json_normalize(data["data"])
    return df

# Date range for 2025
data in Chunks

In [ ]:
start = datetime(2025, 1, 1)
end = datetime(2025, 12, 31)

In [ ]:
# Pull data in chunks
dfs = []
current = start

while current <= end:
    next_date = current + timedelta(days=30)

    if next_date > end:
        next_date = end

    df = get_carbon_intensity(
        current.strftime("%Y-%m-%d"),
        next_date.strftime("%Y-%m-%d")
    )

    dfs.append(df)

    current = next_date
    

# Combine all chunks

In [29]:
df_ci = pd.concat(dfs, ignore_index=True)
df_ci.head()

,from,to,intensity.forecast,intensity.actual,intensity.index
0,2024-12-31T23:30Z,2025-01-01T00:00Z,53.0,51,low
1,2025-01-01T00:00Z,2025-01-01T00:30Z,49.0,55,low
2,2025-01-01T00:30Z,2025-01-01T01:00Z,52.0,54,low
3,2025-01-01T01:00Z,2025-01-01T01:30Z,56.0,53,low
4,2025-01-01T01:30Z,2025-01-01T02:00Z,53.0,53,low


In [30]:
df_ci.columns

Index(['from', 'to', 'intensity.forecast', 'intensity.actual',
       'intensity.index'],
      dtype='str')

# Clean the Dataset
Convert timestamp:

In [31]:
df_ci["timestamp"] = pd.to_datetime(df_ci["from"])

df_ci.head()

,from,to,intensity.forecast,intensity.actual,intensity.index,timestamp
0,2024-12-31T23:30Z,2025-01-01T00:00Z,53.0,51,low,2024-12-31 23:30:00+00:00
1,2025-01-01T00:00Z,2025-01-01T00:30Z,49.0,55,low,2025-01-01 00:00:00+00:00
2,2025-01-01T00:30Z,2025-01-01T01:00Z,52.0,54,low,2025-01-01 00:30:00+00:00
3,2025-01-01T01:00Z,2025-01-01T01:30Z,56.0,53,low,2025-01-01 01:00:00+00:00
4,2025-01-01T01:30Z,2025-01-01T02:00Z,53.0,53,low,2025-01-01 01:30:00+00:00


In [33]:
# Rename columns to remove dots

df_ci = df_ci.rename(columns={
    "intensity.forecast": "intensity_forecast",
    "intensity.actual": "intensity_actual",
    "intensity.index": "intensity_index"
})

df_ci.head()

,from,to,intensity_forecast,intensity_actual,intensity_index,timestamp
0,2024-12-31T23:30Z,2025-01-01T00:00Z,53.0,51,low,2024-12-31 23:30:00+00:00
1,2025-01-01T00:00Z,2025-01-01T00:30Z,49.0,55,low,2025-01-01 00:00:00+00:00
2,2025-01-01T00:30Z,2025-01-01T01:00Z,52.0,54,low,2025-01-01 00:30:00+00:00
3,2025-01-01T01:00Z,2025-01-01T01:30Z,56.0,53,low,2025-01-01 01:00:00+00:00
4,2025-01-01T01:30Z,2025-01-01T02:00Z,53.0,53,low,2025-01-01 01:30:00+00:00


In [36]:
# Create timestamp column
df_ci["timestamp"] = pd.to_datetime(df_ci["from"], utc=True)
df_ci.head()

,from,to,intensity_forecast,intensity_actual,intensity_index,timestamp
0,2024-12-31T23:30Z,2025-01-01T00:00Z,53.0,51,low,2024-12-31 23:30:00+00:00
1,2025-01-01T00:00Z,2025-01-01T00:30Z,49.0,55,low,2025-01-01 00:00:00+00:00
2,2025-01-01T00:30Z,2025-01-01T01:00Z,52.0,54,low,2025-01-01 00:30:00+00:00
3,2025-01-01T01:00Z,2025-01-01T01:30Z,56.0,53,low,2025-01-01 01:00:00+00:00
4,2025-01-01T01:30Z,2025-01-01T02:00Z,53.0,53,low,2025-01-01 01:30:00+00:00


In [40]:
# Strictly for 2025 only

df_ci = df_ci[
    (df_ci["timestamp"] >= "2025-01-01") &
    (df_ci["timestamp"] < "2026-01-01")
]

df_ci.head()

,from,to,intensity_forecast,intensity_actual,intensity_index,timestamp
1,2025-01-01T00:00Z,2025-01-01T00:30Z,49.0,55,low,2025-01-01 00:00:00+00:00
2,2025-01-01T00:30Z,2025-01-01T01:00Z,52.0,54,low,2025-01-01 00:30:00+00:00
3,2025-01-01T01:00Z,2025-01-01T01:30Z,56.0,53,low,2025-01-01 01:00:00+00:00
4,2025-01-01T01:30Z,2025-01-01T02:00Z,53.0,53,low,2025-01-01 01:30:00+00:00
5,2025-01-01T02:00Z,2025-01-01T02:30Z,53.0,47,low,2025-01-01 02:00:00+00:00


# Final Carbon intensity column

Use actual where available, otherwise forecast.

In [42]:
df_ci["carbon_intensity"] = df_ci["intensity_actual"].fillna(df_ci["intensity_forecast"])

df_ci.head()

,from,to,intensity_forecast,intensity_actual,intensity_index,timestamp,carbon_intensity
1,2025-01-01T00:00Z,2025-01-01T00:30Z,49.0,55,low,2025-01-01 00:00:00+00:00,55
2,2025-01-01T00:30Z,2025-01-01T01:00Z,52.0,54,low,2025-01-01 00:30:00+00:00,54
3,2025-01-01T01:00Z,2025-01-01T01:30Z,56.0,53,low,2025-01-01 01:00:00+00:00,53
4,2025-01-01T01:30Z,2025-01-01T02:00Z,53.0,53,low,2025-01-01 01:30:00+00:00,53
5,2025-01-01T02:00Z,2025-01-01T02:30Z,53.0,47,low,2025-01-01 02:00:00+00:00,47


# Final needed columns

In [43]:
df_ci = df_ci[["timestamp", "carbon_intensity"]]
df_ci.head()

,timestamp,carbon_intensity
1,2025-01-01 00:00:00+00:00,55
2,2025-01-01 00:30:00+00:00,54
3,2025-01-01 01:00:00+00:00,53
4,2025-01-01 01:30:00+00:00,53
5,2025-01-01 02:00:00+00:00,47


In [48]:
# Sort values
df_ci = df_ci.sort_values("timestamp").reset_index(drop=True)

In [62]:
# Remove duplicate timestamps
df_ci = df_ci.drop_duplicates(subset="timestamp").reset_index(drop=True)

In [63]:
# Check for missing values
df_ci.isnull().sum()

timestamp           0
carbon_intensity    0
dtype: int64

In [64]:
# Check duplicates
df_ci.duplicated().sum()

np.int64(0)

# Verify Frequency

In [65]:
df_ci["timestamp"].diff().value_counts().head(10)

timestamp
0 days 00:30:00    17471
Name: count, dtype: int64

In [66]:
# dataset shape
df_ci.shape

(17472, 2)

# Preview final cleaned dataset

In [67]:
df_ci.head()

,timestamp,carbon_intensity
0,2025-01-01 00:00:00+00:00,55
1,2025-01-01 00:30:00+00:00,54
2,2025-01-01 01:00:00+00:00,53
3,2025-01-01 01:30:00+00:00,53
4,2025-01-01 02:00:00+00:00,47


# Cleaned file

In [68]:
df_ci.to_csv("carbon_intensity_2025.csv", index=False)